In [1]:
import pandas as pd
import numpy as np
!pip install kagglehub
! pip install keras

In [2]:
import kagglehub

# Download latest version
dataset_path = kagglehub.dataset_download("rifatulmajumder23/combined-unknown-pneumonia-and-tuberculosis")

print("Path to dataset files:", dataset_path)

Path to dataset files: C:\Users\tanee\.cache\kagglehub\datasets\rifatulmajumder23\combined-unknown-pneumonia-and-tuberculosis\versions\1


In [3]:
import os
data_path = os.path.join(dataset_path, "data")

train_dir = os.path.join(data_path, "train")
val_dir= os.path.join(data_path, "val")
test_dir= os.path.join(data_path, "test")


In [4]:
import tensorflow as tf
import keras

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale = 1/255)

C:\Users\tanee\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [5]:
training_set = train_datagen.flow_from_directory(train_dir,
target_size = (64,64),
color_mode = 'grayscale',
class_mode = 'sparse')

Found 13028 images belonging to 4 classes.


In [6]:
training_set.class_indices

{'NORMAL': 0, 'PNEUMONIA': 1, 'TUBERCULOSIS': 2, 'UNKNOWN': 3}

In [7]:
test_datagen = ImageDataGenerator(rescale = 1/255)

testing_set = train_datagen.flow_from_directory(test_dir,
target_size = (64,64),
color_mode = "grayscale",
class_mode = 'sparse')


Found 1527 images belonging to 4 classes.


In [8]:
testing_set.class_indices

{'NORMAL': 0, 'PNEUMONIA': 1, 'TUBERCULOSIS': 2, 'UNKNOWN': 3}

In [9]:
from keras.models import Sequential
from keras.layers import Conv2D

cnn_model = Sequential()

cnn_model.add(Conv2D(input_shape = [64,64,1], filters = 32, kernel_size= 3, activation = 'relu') )

C:\Users\tanee\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
from keras.layers import MaxPooling2D

cnn_model.add(MaxPooling2D(pool_size=2, strides=2))

In [14]:
from keras.layers import Flatten

cnn_model.add(Flatten())

In [15]:
from keras.layers import Dense

cnn_model.add(Dense(units = 128, activation = 'relu') )
cnn_model.add(Dense(units = 4, activation = 'softmax'))

cnn_model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ["Accuracy"])

In [16]:
cnn_model.fit(x = training_set, validation_data = testing_set, epochs = 25)

Epoch 1/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 513s 1s/step - Accuracy: 0.7209 - loss: 0.6546 - val_Accuracy: 0.7944 - val_loss: 0.5045
Epoch 2/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 568s 1s/step - Accuracy: 0.8506 - loss: 0.3714 - val_Accuracy: 0.8815 - val_loss: 0.2904
Epoch 3/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 681s 2s/step - Accuracy: 0.8861 - loss: 0.2878 - val_Accuracy: 0.8454 - val_loss: 0.3606
Epoch 4/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 671s 2s/step - Accuracy: 0.9032 - loss: 0.2500 - val_Accuracy: 0.9037 - val_loss: 0.2584
Epoch 5/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 662s 2s/step - Accuracy: 0.9123 - loss: 0.2216 - val_Accuracy: 0.9037 - val_loss: 0.2498
Epoch 6/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 723s 2s/step - Accuracy: 0.9224 - loss: 0.1988 - val_Accuracy: 0.9057 - val_loss: 0.2305
Epoch 7/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 713s 2s/step - Accuracy: 0.9308 - loss: 0.1820 - val_Accuracy: 0.9149 - val_loss: 0.2314
Epoch 8/25
408/408 ━━━━━━━━━━━━━━━━━━━━ 747s 2s/step - Accuracy: 0.9402 - loss: 0.1575 - val_Accu

In [1]:
from PIL import Image

# Test_image_1 (PNEUMONIA)
#test_image = Image.open(r"D:\tb\s-75.png")

test_image = Image.open(r"D:\tb\person64.jpeg")

#test_image = Image.open(r"D:\tb\r-680.png")

test_image = test_image.convert('L')
test_image = test_image.resize((64,64))
test_image = np.array(test_image)
test_image = test_image.astype('float32') /255
test_image = np.expand_dims(test_image, axis=0)


result = cnn_model.predict(test_image)

class_labels = {0: 'NORMAL', 1: 'PNEUMONIA', 2: 'TUBERCULOSIS', 3: 'UNKNOWN'} # Labels

predicted_class_index = np.argmax(result)
confidence = np.max(result) * 100

print("Predicted Class:", class_labels[predicted_class_index])
print("Confidence:", confidence, "%")

NameError: name 'np' is not defined

In [22]:
from joblib import dump

dump(cnn_model,"XRay_Model.joblib")

['XRay_Model.joblib']

In [29]:
from joblib import load
import numpy as np
from PIL import Image

model = load("XRay_Model.joblib")

test_image = Image.open(r"D:\tb\person64.jpeg")

test_image = test_image.convert('L')
test_image = test_image.resize((64,64))
test_image = np.array(test_image)
test_image = test_image.astype('float32')/255
test_image = np.expand_dims(test_image, axis =0)

result = model.predict(test_image)
print(result)

predicted_class_index = np.argmax(result)

classes = {0: 'NORMAL' , 1: "PNEUMONIA", 2: "TUBERCULOSIS", 3: "UNKNOWN"}

print("Predicted Problem => ",classes[predicted_class_index])
print("Confidence Percentage => ", np.max(result)*100,"%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
[[6.9319635e-06 9.9999177e-01 9.8562214e-07 3.2073655e-07]]
Predicted Problem =>  PNEUMONIA
Confidence Percentage =>  99.999176 %


In [30]:
cnn_model.save("XRay_Model.keras")

In [1]:
from joblib import load
import numpy as np
from PIL import Image

model = load("XRay_Model.joblib")

test_image = Image.open(r"D:\tb\person64.jpeg")

test_image = test_image.convert('L')
test_image = test_image.resize((64,64))
test_image = np.array(test_image)
test_image = test_image.astype('float32')/255
test_image = np.expand_dims(test_image, axis =0)

result = model.predict(test_image)
print(result)

predicted_class_index = np.argmax(result)

classes = {0: 'NORMAL' , 1: "PNEUMONIA", 2: "TUBERCULOSIS", 3: "UNKNOWN"}

print("Predicted Problem => ",classes[predicted_class_index])
print("Confidence Percentage => ", np.max(result)*100,"%")

C:\Users\tanee\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 503ms/step
[[6.9319967e-06 9.9999177e-01 9.8560986e-07 3.2073348e-07]]
Predicted Problem =>  PNEUMONIA
Confidence Percentage =>  99.999176 %


In [31]:
import numpy as np
from PIL import Image
import tensorflow as tf
import os

app = Flask(__name__)

cnn_model = tf.keras.models. load_model(r"E:\Generative AI\AI Projects\X Ray Analysis\XRay_Model.keras")

CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis", "Invalid Image"]

UPLOAD_FOLDER = "static/uploads"from flask import Flask, render_template, request


os.makedirs(UPLOAD_FOLDER, exist_ok=True)

def preprocess_image(image_path):
img = Image.open(image_path).convert("L")
img = img.resize((64, 64))
img = np.array(img)
img = img.astype("float32") / 255.0
img = np.expand_dims(img, axis=0)
return img

@app. route("/")
def home():
return render_template("XRay.html") # ---- > HTML TO RUN THIS PROJECT

@app.route("/predict", methods=["POST"])
def predict ():
if "xray" not in request.files:

SyntaxError: invalid syntax (1886797072.py, line 12)